In [1]:
# =============================================================================
# GUS01F: Loading Numerical Data onto GeoTERYT Records
# =============================================================================
# This notebook demonstrates the v4.0 data storage capabilities:
# 1. Load BDL demographic data (subject P2137 - population)
# 2. Process and attach time series to TERYTRecord objects
# 3. Query data on individual records
# 4. Aggregate for regions (voivodeships)
# 5. Produce joint/marginal distributions
# 6. Save/reload database with data persistence
# =============================================================================

# STEP 1: Imports and Path Setup
import os
import sys
from pathlib import Path
import importlib
import gc

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()
tools_path = repo_root / 'Code' / 'tools'
if str(tools_path) not in sys.path:
    sys.path.insert(0, str(tools_path))

# Reload geoTERYT_db to get latest version (v4.0)
import geoTERYT_db as gtdb
importlib.reload(gtdb)

# Data paths
data_root = repo_root.parent.parent / 'Data'
geo_root = data_root / 'Geospatial'
gus_root = data_root / 'GUS'

print(f"Repository root: {repo_root}")
print(f"Data root: {data_root}")
print(f"GUS root: {gus_root}")

Repository root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
Data root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS


In [2]:
# =============================================================================
# STEP 2: Load Complete GeoTERYT Database
# =============================================================================
complete_db_path = geo_root / 'geoteryt_complete_geom_OW.pkl'
db = gtdb.load_complete_database(complete_db_path)
db.print_summary()

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_geom_OW.pkl...
  Database version: 3.1
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4560 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3612
  ✓ Records with old_woj: 2658
GeoTERYT Database Summary (v3.0)
Total records:           4,560
Year range:              1999 - 2024
------------------------------------------------------------
Administrative levels:
  Voivodeships (2):      16
  Powiats (5):           382
  Gminas (6):            4162
------------------------------------------------------------
Change tracking:
  Records with changes:      772
  Records with level changes: 0
  Records with kind changes:  0
------------------------------------------------------------
Geometry:
  Records with geometr

In [ ]:
# =============================================================================
# STEP 2B: Link Children to Parents (must happen before data loading)
# =============================================================================
db.link_children_to_parents()

In [3]:
# =============================================================================
# STEP 3: Load BDL Source Data
# =============================================================================
df_demographic = pd.read_csv(gus_root / "data" / 'bdl_demographic_data.csv', encoding='utf-8')
df_c_1988 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP1988_data.csv', encoding='utf-8')
df_c_2002 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2002_data.csv', encoding='utf-8')
df_c_2011 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2011_data.csv', encoding='utf-8')
df_c_2021 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2021_data.csv', encoding='utf-8')

df_variables = pd.read_csv(gus_root / "metadata" / 'bdl_variables_level6.csv', encoding='utf-8')
df_c_variables = pd.read_csv(gus_root / "metadata" / 'census_meta.csv', encoding='utf-8')

print(f"df_demographic: {df_demographic.shape}")
print(f"df_variables: {df_variables.shape}")
print(f"\nAvailable subjects: {sorted(df_demographic['subjectId'].unique())}")

df_demographic: (431358, 5)
df_variables: (27922, 10)

Available subjects: ['P1336', 'P2137', 'P2914']


In [4]:
# Edit years for census of 1988

for id, row in df_c_1988.iterrows():
    df_c_1988.at[id, 'values']= df_c_1988.at[id, 'values'].replace(", 'year': '1998'", ", 'year': '1988'")
    
df_c_variables['years'] = df_c_variables['years'].str.replace("[1998]", "[1988]")

# Unify the meta dataframe structure
df_c_variables['n4'] = None
df_c_variables['n5'] = None


In [ ]:
display(df_c_1988)
display(df_demographic)

In [5]:
# Collect all subject ids
subject_ids = {"BDL": [], "Census": {"1988" : [], "2002": [], "2011": [], "2021": []}}

subject_ids["BDL"] = list(df_demographic['subjectId'].unique())
subject_ids["Census"]["1988"] = list(df_c_1988['subjectId'].unique())
subject_ids["Census"]["2002"] = list(df_c_2002['subjectId'].unique())
subject_ids["Census"]["2011"] = list(df_c_2011['subjectId'].unique())
subject_ids["Census"]["2021"] = list(df_c_2021['subjectId'].unique())

subject_ids_flat = subject_ids['BDL'] + subject_ids["Census"]["1988"] + subject_ids["Census"]["2002"] \
    + subject_ids["Census"]["2011"] + subject_ids["Census"]["2021"]

subject_names_dict = {}
for subject in subject_ids_flat:
    subject_names_dict[subject] = ''
    
subject_ids

{'BDL': ['P1336', 'P2137', 'P2914'],
 'Census': {'1988': ['P2884', 'P2885', 'P2883', 'P2887'],
  '2002': ['P2114', 'P2403', 'P2402', 'P2871'],
  '2011': ['P3304', 'P3311', 'P3309', 'P3310', 'P3420'],
  '2021': ['P4253', 'P4320', 'P4345', 'P4287']}}

In [6]:
# Names of different subjects

subject_names_dict['P1336'] = 'pop__sex_URsplit'
subject_names_dict['P2137'] = 'pop__age_sex'
subject_names_dict['P2914'] = 'pop__sex_cities'

subject_names_dict['P2884'] = 'pop__age'
subject_names_dict['P2885'] = 'pop__educ'
subject_names_dict['P2883'] = 'pop__sex'
subject_names_dict['P2887'] = 'hh_size'

subject_names_dict['P2114'] = 'pop__age_sex'
subject_names_dict['P2403'] = 'pop__age_educ'
subject_names_dict['P2402'] = 'pop__sex_educ'
subject_names_dict['P2871'] = 'hh_size'

subject_names_dict['P3304'] = 'pop__age_sex'
subject_names_dict['P3311'] = 'pop__age_educ'
subject_names_dict['P3309'] = 'pop__sex_educ'
subject_names_dict['P3310'] = 'pop__educ_URsplit'
subject_names_dict['P3420'] = 'hh_size'

subject_names_dict['P4253'] = 'pop__age_sex'
subject_names_dict['P4320'] = 'pop__age_educ'
subject_names_dict['P4345'] = 'pop__sex_educ_URsplit'
subject_names_dict['P4287'] = 'hh_size'

subject_names_dict

{'P1336': 'pop__sex_URsplit',
 'P2137': 'pop__age_sex',
 'P2914': 'pop__sex_cities',
 'P2884': 'pop__age',
 'P2885': 'pop__educ',
 'P2883': 'pop__sex',
 'P2887': 'hh_size',
 'P2114': 'pop__age_sex',
 'P2403': 'pop__age_educ',
 'P2402': 'pop__sex_educ',
 'P2871': 'hh_size',
 'P3304': 'pop__age_sex',
 'P3311': 'pop__age_educ',
 'P3309': 'pop__sex_educ',
 'P3310': 'pop__educ_URsplit',
 'P3420': 'hh_size',
 'P4253': 'pop__age_sex',
 'P4320': 'pop__age_educ',
 'P4345': 'pop__sex_educ_URsplit',
 'P4287': 'hh_size'}

In [ ]:
# =============================================================================
# STEP 4: Process Subjects
# =============================================================================
# Use the new process_subject_data() static method on GeoTERYTDatabase

# Example: Process subject P2137 (Population Data) from BDL
subject_id = 'P2137'
df_p2137 = gtdb.GeoTERYTDatabase.process_subject_data(df_demographic, df_variables, subject_id)

print(f"Processed P2137: {df_p2137.shape}")
print(f"\nColumns: {list(df_p2137.columns)}")
print(f"\nCategory columns present:")
for col in ['n1', 'n2', 'n3', 'n4', 'n5']:
    if col in df_p2137.columns:
        print(f"  {col}: {sorted(df_p2137[col].dropna().unique())}")
print(f"\nYears: {sorted(df_p2137['year'].dropna().astype(str).unique())}")
print(f"Unique TERYT IDs: {df_p2137['teryt_id'].nunique()}")
df_p2137.head()

In [ ]:
# Example: Process subject P2114 (Population Data) from Census data 2002

In [7]:
# =============================================================================
# STEP 4: Process ALL Subjects
# =============================================================================

df_subjects = {
    "BDL": df_demographic,
    "Census": {
        "1988": df_c_1988,
        "2002": df_c_2002,
        "2011": df_c_2011,
        "2021": df_c_2021
    }
}

df_processed_subjects = {
    "BDL": {},
    "Census": {
        "1988": {},
        "2002": {},
        "2011": {},
        "2021": {}
    }
}

# P1336 filter: keep only 'miejsce zamieszkania' in n2
# and 'stan na 30 czerwca' in n3
P1336_FILTERS = {'n2': 'miejsce zamieszkania', 'n3': 'stan na 30 czerwca'}

for subject in subject_ids.items():
    if subject[0] == "BDL":
        subjects = subject[1]
        for s in subjects:
            print(f"Processing BDL subject: {s}...")
            df = gtdb.GeoTERYTDatabase.process_subject_data(df_demographic, df_variables, s)
            # Apply P1336 filter
            if s == 'P1336':
                df = gtdb.GeoTERYTDatabase.filter_subject_data(df, s, P1336_FILTERS)
                print(f"  Filtered P1336: {df.shape}")
            df_processed_subjects["BDL"][s] = df
    else:
        for sub in subject[1].items():
            print(f"Processing Census subject: {sub[0]} - {sub[1]}...")
            year = sub[0]
            df_c = df_subjects["Census"][year]
            for s in sub[1]:
                print(f"  Processing subject: {s}...")
                df = gtdb.GeoTERYTDatabase.process_subject_data(df_c, df_c_variables, s)
                df_processed_subjects["Census"][year][s] = df

del df_subjects
gc.collect()

# Save df_processed_subjects for later use
import pickle
with open(gus_root / 'data' / 'processed.pkl', 'wb') as f:
    pickle.dump(df_processed_subjects, f)

Processing BDL subject: P1336...
  Filtered P1336: (1171935, 11)
Processing BDL subject: P2137...
Processing BDL subject: P2914...
Processing Census subject: 1988 - ['P2884', 'P2885', 'P2883', 'P2887']...
  Processing subject: P2884...
  Processing subject: P2885...
  Processing subject: P2883...
  Processing subject: P2887...
Processing Census subject: 2002 - ['P2114', 'P2403', 'P2402', 'P2871']...
  Processing subject: P2114...
  Processing subject: P2403...
  Processing subject: P2402...
  Processing subject: P2871...
Processing Census subject: 2011 - ['P3304', 'P3311', 'P3309', 'P3310', 'P3420']...
  Processing subject: P3304...
  Processing subject: P3311...
  Processing subject: P3309...
  Processing subject: P3310...
  Processing subject: P3420...
Processing Census subject: 2021 - ['P4253', 'P4320', 'P4345', 'P4287']...
  Processing subject: P4253...
  Processing subject: P4320...
  Processing subject: P4345...
  Processing subject: P4287...


In [8]:
# Open the saved processed data to verify
import pickle
with open(gus_root / 'data' / 'processed.pkl', 'rb') as f:
    df_processed_subjects = pickle.load(f)
    

In [9]:
# =============================================================================
# STEP 5: Load Subject Data onto TERYTRecords
# =============================================================================
# This attaches time series data to each matching TERYTRecord in the database

for subject in subject_ids.items():
    if subject[0] == "BDL":
        subjects = subject[1]
        for s in subjects:
            print(f"Loading BDL subject: {s}...")
            df = df_processed_subjects["BDL"][s]
            stats = db.load_subject_data(df, source_type='BDL', subject_id=s, subject_name=subject_names_dict[s])
            print(f"  Loading statistics: {stats['matched_teryts']} matched, {stats['unmatched_teryts']} unmatched")
            print(f"  Total data points loaded: {stats['total_data_points']:,}")
    else:
        for sub in subject[1].items():
            print(f"Loading Census subject: {sub[0]} - {sub[1]}...")
            year = sub[0]
            for s in sub[1]:
                print(f"  Loading subject: {s}...")
                df = df_processed_subjects["Census"][year][s]
                stats = db.load_subject_data(df, source_type='Census', subject_id=s, subject_name=subject_names_dict[s])
                print(f"    Loading statistics: {stats['matched_teryts']} matched, {stats['unmatched_teryts']} unmatched")
                print(f"    Total data points loaded: {stats['total_data_points']:,}")

# Free memory
del df_processed_subjects
gc.collect()

Loading BDL subject: P1336...
  ✓ Loaded 1,169,136 data points for subject P1336
  ✓ Matched 4569 TERYT records, 10 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0000000', '0216001', '0410001', '1207132', '1210001', '1431981', '1431991', '1465158', '1465998', '2002162']
  Loading statistics: 4569 matched, 10 unmatched
  Total data points loaded: 1,169,136
Loading BDL subject: P2137...
  ✓ Loaded 7,338,336 data points for subject P2137
  ✓ Matched 4548 TERYT records, 8 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0000000', '0216001', '0410001', '1210001', '1431981', '1431991', '1465158', '1465998']
  Loading statistics: 4548 matched, 8 unmatched
  Total data points loaded: 7,338,336
Loading BDL subject: P2914...
  ✓ Loaded 1,505,856 data points for subject P2914
  ✓ Matched 4548 TERYT records, 8 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0000000', '0216001', '0410001', '1210001', '1431981', '1431991', '1465158', '1465998']
  Loading statistics: 4548 matched, 8 unmatched
  Tota

0

In [ ]:
# =============================================================================
# STEP 6: Create Merged Subjects (unified census + BDL time series)
# =============================================================================
# Creates NEW merged subjects (M_ prefix) from groups sharing the same topic.
# Original subjects are NOT modified - raw data stays intact.
# For age dimensions: computes unified bins via common break points.
# For sex dimensions: exact label matching.

importlib.reload(gtdb)

print("Creating merged subjects...")
merged_info = db.create_merged_subjects(subject_names_dict)

# Update subject_names_dict with merged subjects
for merged_sid, source_ids in merged_info.items():
    group_name = merged_sid.replace('M_', '')
    subject_names_dict[merged_sid] = group_name
    print(f"  Added {merged_sid} -> '{group_name}'")

# Show data summary after merge
summary = db.get_data_summary()
print(f"\nAfter merge:")
print(f"  Records with data: {summary['records_with_data']}")
print(f"  Subjects: {summary['n_subjects']} ({summary['subjects']})")
print(f"  Total data series: {summary['total_data_series']:,}")
print(f"  Total data points: {summary['total_data_points']:,}")

Unifying census subjects...
Found 4 subject groups to unify:
  pop__age_sex: ['P2137', 'P2114', 'P3304', 'P4253']
  hh_size: ['P2887', 'P2871', 'P3420', 'P4287']
  pop__age_educ: ['P2403', 'P3311', 'P4320']
  pop__sex_educ: ['P2402', 'P3309']
  ✓ Unified ['P2114', 'P3304', 'P4253'] → P2137: 11145 records, 657759 series merged
  ✓ Unified ['P2871', 'P3420', 'P4287'] → P2887: 7824 records, 80212 series merged
  ✓ Unified ['P3311', 'P4320'] → P2403: 759 records, 85383 series merged
  ✓ Unified ['P3309'] → P2402: 379 records, 10233 series merged

Mapping: {'P2114': 'P2137', 'P3304': 'P2137', 'P4253': 'P2137', 'P2871': 'P2887', 'P3420': 'P2887', 'P4287': 'P2887', 'P3311': 'P2403', 'P4320': 'P2403', 'P3309': 'P2402'}

After unification:
  Records with data: 4532
  Subjects: 11 (['P1336', 'P2137', 'P2402', 'P2403', 'P2883', 'P2884', 'P2885', 'P2887', 'P2914', 'P3310', 'P4345'])
  Total data points: 10,824,393


In [11]:
# =============================================================================
# STEP 7: Extract Total Population & Classify Urban/Rural
# =============================================================================

print("Extracting total population...")
n_pop = db.extract_population(subject_names_dict)

print("\nClassifying urban/rural...")
n_class = db.classify_population()

Extracting total population...
  ✓ Extracted population for 4532 records

Classifying urban/rural...
  ✓ Classified 3411 records by urban/rural


In [12]:
# =============================================================================
# STEP 8: Code Dimension Labels
# =============================================================================

print("Coding dimension labels...")
n_coded = db.code_dimension_labels(subject_names_dict)

Coding dimension labels...
  ✓ Coded dimension labels for 961772 DataSeries across 11 subjects


In [13]:
# =============================================================================
# STEP 9: Save Database with All Data
# =============================================================================

save_path = geo_root / 'geoteryt_complete_final.pkl'
db.save_complete(save_path)

Saving complete database to /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl...
  ✓ Saved 4560 records
  ✓ Records with data: 4532
  ✓ File size: 1403.4 MB
  ✓ Path: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl


In [ ]:
# =============================================================================
# STEP 10: Verify Data on Individual Records
# =============================================================================
data_summary = db.get_data_summary()
print("Data Summary:")
for k, v in data_summary.items():
    print(f"  {k}: {v}")

# Show subjects and their types
print("\nSubjects:")
for sid in sorted(data_summary['subjects']):
    sname = subject_names_dict.get(sid, '')
    prefix = "MERGED" if sid.startswith('M_') else "RAW"
    print(f"  [{prefix}] {sid}: {sname}")